# Quantum-circuit preprocessing for RL placement

The production implementation lives in `rl_qtranspiler.preprocessing`. This notebook imports that canonical implementation so its data types cannot diverge from the training package.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator

from rl_qtranspiler.preprocessing import (
    mapped_one_qubit_gates_for_slot,
    preprocess_for_swap_routing,
    restore_without_routing,
)


## Example

Only adjacent two-qubit gates are fused. Removed one-qubit gates retain their logical qubit and insertion slot.

In [ ]:
circuit = QuantumCircuit(3, name="example")
circuit.h(0)
circuit.cx(0, 1)
circuit.cz(1, 0)
circuit.rz(0.3, 2)
circuit.cx(1, 2)
circuit.x(1)
circuit.cx(1, 2)

result = preprocess_for_swap_routing(circuit)
restored = restore_without_routing(result)

print(result.backbone)
print(*result.one_qubit_gates, sep="\n")
assert len(result.one_qubit_gates) == 3
assert len(result.instructions) == 3
assert result.instructions[0].source_indices == (1, 2)
assert Operator(circuit).equiv(Operator(restored))


## Restoring gates after routing

At each stored slot, call `mapped_one_qubit_gates_for_slot(result, slot, logical_to_physical)` with the router's current mapping. This preserves the one-qubit gates without putting them into the placement agent's interaction graph.